In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
from scipy.stats import wilcoxon

plt.style.use("default")

plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial"],
        "font.size": 6,
        "axes.titlesize": 7,
        "axes.labelsize": 6,
        "axes.linewidth": 0.8,
        "xtick.labelsize": 6,
        "ytick.labelsize": 6,
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "legend.fontsize": 6,
        "legend.frameon": False,
        "lines.linewidth": 1.2,
        "lines.markersize": 4,
        "figure.figsize": (3.5, 2.6),
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "savefig.bbox": "standard",
        # "savefig.pad_inches": 0.02,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "mathtext.fontset": "dejavusans",
    }
)
cm = 1 / 2.54  # centimeters in inches

results_path = "trafficking_inference/method_testing/results/results_median_accs/"
figure_path = "trafficking_inference/method_testing/figures/results_median_accs/"
# if the results_path does not exist, create it


if not os.path.exists(figure_path):
    os.makedirs(figure_path)

method_names = [
    "StitchOT $\\tau_a=0.95$",
    "StitchOT $\\tau_a=0.8$",
    "StitchOT $\\tau_a=0.5$",
    "StitchOT $\\tau_a=0.5$, \n no resc.",
    "StitchOT $\\tau_a=0.5$,\n no subsamp.",
    "StitchOT $\\tau_a=0.5$,\n source & target low S1PR1",
    "StitchOT $\\tau_a=0.5$,\n target low S1PR1",
    "StitchOT $\\tau_a=0.5$,\n true fractions",
    "Random",
    "JointOT $\\tau_b=0.95$",
    "JointOT $\\tau_b=0.8$",
    "JointOT $\\tau_b=0.5$",
    "JointOT $\\tau_b=0.5$,\n true fractions",
    "JointOT $\\tau_b=0.5$,\n no subsamp.",
    "JointOT $\\tau_b=0.5$,\n source & target low S1PR1",
    "JointOT $\\tau_b=0.5$,\n target low S1PR1",
]

stats_names = [
    "True Accuracy",
    "True thres. Accuracy",
    "Rel. Accuracy",
    "Rel. avg. thres. Accuracy",
    "fate matrix(true probs)",
    "thresh. 1/3 (true probs)",
    "thresh. 1/2 (true probs)",
    "argmax (true probs)",
    "rawOT weights (inferred)",
    "fate matrix (inferred)",
    "thresh. 1/3 (inferred)",
    "thresh. 1/2 (inferred)",
    "argmax (inferred)",
    "Random",
    "TV distance transition probs.",
    "IAE",
]

# generate colors
# for the eight stitchOT methods use sequential colormaps blue
# for random use colors_3[2]
# for the eight jointOT methods use sequential colormaps orange
colors_3 = [
    "#E64B35FF",
    "#4DBBD5FF",
    "#b3b3b3",
]  # 1 'salmon','turquoise', #2 "teal", "mediumorchid" #'paleturquoise', 'lightsteelblue' ,#"gray","lightgray"
colors = []
for i in range(8):
    colors.append(sns.color_palette("Reds", 10)[i + 1])
colors.append(colors_3[2])
for i in range(7):
    colors.append(sns.color_palette("Blues", 10)[i + 2])


results = np.zeros((10000, 16, 16))
real_fractions = np.zeros((10000, 4))
for job_id in range(100):
    try:
        results[job_id * 100 : (job_id + 1) * 100, :, :] = np.load(
            f"{results_path}/results_{job_id}.npy"
        )
        real_fractions[job_id * 100 : (job_id + 1) * 100, :] = np.load(
            f"{results_path}/real_fractions_{job_id}.npy"
        )
    except FileNotFoundError:
        print(f"File results_{job_id}.npy not found. Skipping this job.")
        continue


# delete all rows that only contain zeros
real_fractions = real_fractions[~np.all(results == 0, axis=(1, 2))]
results = results[~np.all(results == 0, axis=(1, 2))]

# how many rows are left
print(f"Number of rows left: {results.shape[0]}")
# delete rows containing nans
real_fractions = real_fractions[~np.all(np.isnan(results), axis=(1, 2))]
results = results[~np.all(np.isnan(results), axis=(1, 2))]

print(f"Number of rows left after removing nans: {results.shape[0]}")
# delete all rows where any of real_fractions is <100
# results = results[~np.any(real_fractions < 100, axis=1)]
print(
    f"Number of rows left after removing rows with real_fractions < 100: {results.shape[0]}"
)
# real_fractions = real_fractions[~np.any(real_fractions < 100, axis=1)]

Number of rows left: 4233
Number of rows left after removing nans: 4233
Number of rows left after removing rows with real_fractions < 100: 4233


In [ ]:
# Do boxplots for all methods,
for metric_id in [3, 9, 14, 15]:
    plt.figure(figsize=((12, 6, "cm")), layout="constrained")
    results_metric = results[~np.isnan(results[:, :, metric_id]).any(axis=1)]
    sns.violinplot(
        data=results_metric[:, :, metric_id],
        palette=colors,
        linewidth=1,
        inner="quart",
        cut=0,
    )
    # also plot a white ot with grey edge for the mean
    plt.scatter(
        x=np.arange(len(method_names)),
        y=[np.mean(results_metric[:, i, metric_id]) for i in range(len(method_names))],
        color="white",
        edgecolor="grey",
        s=3,
        zorder=10,
        linewidth=0.4,
    )
    # annotate the boxplots with the method names
    plt.xticks(ticks=np.arange(len(method_names)), labels=method_names, rotation=90)
    plt.ylim(-0.02, None)
    plt.yticks(ticks=np.arange(0, 1.1, 0.2))
    # change xticks size to 5.5
    plt.xticks(fontsize=6)
    # add horizontal grid lines
    plt.grid(axis="y", linestyle="solid", alpha=0.4)
    plt.gca().set_axisbelow(True)
    # plt.ylabel(stats_names[metric_id])
    plt.savefig(
        f"{figure_path}violin_{stats_names[metric_id].replace(' ', '_')}_all_methods.pdf"
    )
    plt.close()

# do a boxplot only comparing StitchOT tau_a=0.5 and JointOT tau_b=0.5 adn Random, for Rel. thres. Accuracy (index 3)
plt.figure(figsize=(2.4, 4.8, "cm"), layout="constrained")
sns.violinplot(
    data=results[:, [2, 11, 8], 3], linewidth=1, palette=colors_3, inner="quart", cut=0
)
# also plot a white ot with grey edge for the mean
plt.scatter(
    x=[0, 1, 2],
    y=[
        np.mean(results[:, 2, 3]),
        np.mean(results[:, 11, 3]),
        np.mean(results[:, 8, 3]),
    ],
    color="white",
    edgecolor="grey",
    s=4,
    zorder=10,
    linewidth=0.4,
)
plt.gca().set_position([0.23, 0.23, 0.75, 0.70])
# also color the area above y>1.0 in light grey
# take x values from current axis limits
plt.fill_between(
    x=[plt.gca().get_xlim()[0], plt.gca().get_xlim()[1]],
    y1=1.0,
    y2=2,
    color="lightgrey",
    alpha=1,
    zorder=0,
    edgecolor=None,
)
# compute p values between the first two boxplots (StitchOT tau_a=0.5 and JointOT tau_b=0.5) using a t-test
stitchOT_results = results[:, 2, 3]
jointOT_results = results[:, 11, 3]


stat, p = wilcoxon(stitchOT_results, jointOT_results)
print("Wilcoxon Test for rel. thres. Accuracy between StitchOT and JointOT:")
print(f"Wilcoxon test between StitchOT and JointOT: statistic={stat}, p-value={p}")

mean_diff = np.mean(stitchOT_results - jointOT_results)
print(f"Mean of paired differences (StitchOT - JointOT): {mean_diff} \n")
# annotate the boxplots with the method names
plt.xticks(ticks=np.arange(3), labels=["StitchOT", "JointOT", "Random"], rotation=90)
plt.ylim(-0.02, 1.22)
plt.xlim(-0.5, 2.5)
plt.gca().set_position([0.23, 0.23, 0.75, 0.74])
# set yticks to be 0, 0.2, 0.4, 0.6, 0.8, 1
plt.yticks(ticks=np.arange(0, 1.1, 0.2))
# plt.ylabel(r'$\overline{\text{ACC}^{\text{R}}_{\psi}}$')
plt.grid(axis="y", linestyle="solid", alpha=0.4)
plt.gca().set_axisbelow(True)
plt.savefig(f"{figure_path}violin_rel_thres_accs_stitchOT_vs_jointOT_vs_random.pdf")
plt.close()


# boxplot only comparing StitchOT tau_a=0.5 and JointOT tau_b=0.5 adn Random, for Rel. thres. Accuracy (index 3)
plt.figure(figsize=(2.4, 4.8, "cm"), layout="constrained")
sns.violinplot(
    data=results[:, [2, 11, 8], 9], palette=colors_3, linewidth=1, inner="quart", cut=0
)
plt.scatter(
    x=[0, 1, 2],
    y=[
        np.mean(results[:, 2, 9]),
        np.mean(results[:, 11, 9]),
        np.mean(results[:, 8, 9]),
    ],
    color="white",
    edgecolor="grey",
    s=4,
    zorder=10,
    linewidth=0.4,
)
plt.xticks(ticks=np.arange(3), labels=["StitchOT", "JointOT", "Random"], rotation=90)
plt.gca().set_position([0.23, 0.23, 0.75, 0.74])
plt.ylim(-0.02, 1.0)
plt.yticks(ticks=np.arange(0, 1.1, 0.2))
plt.grid(axis="y", linestyle="solid", alpha=0.4)
# make sure its behind the violinplot
plt.gca().set_axisbelow(True)
plt.savefig(
    f"{figure_path}violin_tv_distance_transition_rates_stitchOT_vs_jointOT_vs_random.pdf"
)
plt.close()
print(
    "Wilcoxon Test for TV distance Rel. transition rates between StitchOT and JointOT:"
)
stitchOT_results = results[:, 2, 9]
jointOT_results = results[:, 11, 9]
stat, p = wilcoxon(stitchOT_results, jointOT_results)
print(f"Wilcoxon test between StitchOT and JointOT: statistic={stat}, p-value={p}")
mean_diff = np.mean(stitchOT_results - jointOT_results)
print(f"Mean of paired differences (StitchOT - JointOT): {mean_diff} \n")

# same but for IEA (index 15)
plt.figure(figsize=(2.4, 4.8, "cm"), layout="constrained")
sns.violinplot(
    data=results[:, [2, 11, 8], 15], palette=colors_3, linewidth=1, inner="quart", cut=0
)
plt.scatter(
    x=[0, 1, 2],
    y=[
        np.mean(results[:, 2, 15]),
        np.mean(results[:, 11, 15]),
        np.mean(results[:, 8, 15]),
    ],
    color="white",
    edgecolor="grey",
    s=4,
    zorder=10,
    linewidth=0.4,
)
plt.gca().set_position([0.23, 0.23, 0.75, 0.74])
plt.xticks(ticks=np.arange(3), labels=["StitchOT", "JointOT", "Random"], rotation=90)
plt.ylim(-0.02, 1.0)
plt.yticks(ticks=np.arange(0, 1.1, 0.2))
plt.grid(axis="y", linestyle="solid", alpha=0.4)
plt.gca().set_axisbelow(True)
plt.savefig(f"{figure_path}violin_IAE_stitchOT_vs_jointOT_vs_random.pdf")
plt.close()
print("Wilcoxon Test for IEA between StitchOT and JointOT:")
stitchOT_results = results[:, 2, 15]
jointOT_results = results[:, 11, 15]
stat, p = wilcoxon(stitchOT_results, jointOT_results)
print(f"Wilcoxon test between StitchOT and JointOT: statistic={stat}, p-value={p}")
mean_diff = np.mean(stitchOT_results - jointOT_results)
print(f"Mean of paired differences (StitchOT - JointOT): {mean_diff} \n")


# same but for TV distance transition probs. (index 14)
plt.figure(figsize=(2.4, 4.8, "cm"), layout="constrained")
sns.violinplot(
    data=results[:, [2, 11, 8], 14], palette=colors_3, linewidth=1, inner="quart", cut=0
)
# rmove rows that have nans in either stitchOT or jointOT results
results_no_nan = results[
    ~np.isnan(results[:, 2, 14]) & ~np.isnan(results[:, 11, 14]), :, :
]
# print how many nans were removed
print(
    f"Removed {results.shape[0] - results_no_nan.shape[0]} rows with nans in either StitchOT or JointOT results for TV distance transition probs."
)
print(
    [
        np.mean(results_no_nan[:, 2, 14]),
        np.mean(results_no_nan[:, 11, 14]),
        np.mean(results_no_nan[:, 8, 14]),
    ]
)
plt.scatter(
    x=[0, 1, 2],
    y=[
        np.mean(results_no_nan[:, 2, 14]),
        np.mean(results_no_nan[:, 11, 14]),
        np.mean(results_no_nan[:, 8, 14]),
    ],
    color="white",
    edgecolor="grey",
    s=4,
    zorder=10,
    linewidth=0.4,
)
plt.gca().set_position([0.23, 0.23, 0.75, 0.74])
plt.xticks(ticks=np.arange(3), labels=["StitchOT", "JointOT", "Random"], rotation=90)
plt.yticks(ticks=np.arange(0, 1.1, 0.2))
plt.ylim(-0.02, 1.0)
plt.grid(axis="y", linestyle="solid", alpha=0.4)
plt.gca().set_axisbelow(True)
plt.savefig(
    f"{figure_path}violin_tv_distance_transition_probs_stitchOT_vs_jointOT_vs_random.pdf"
)
plt.close()
print("Wilcoxon Test for TV distance transition probs. between StitchOT and JointOT:")
stitchOT_results = results[:, 2, 14]
jointOT_results = results[:, 11, 14]
# remove nans from both arrays
valid_idx = ~np.isnan(stitchOT_results) & ~np.isnan(jointOT_results)
stitchOT_results = stitchOT_results[valid_idx]
jointOT_results = jointOT_results[valid_idx]
stat, p = wilcoxon(stitchOT_results, jointOT_results)
print(f"Wilcoxon test between StitchOT and JointOT: statistic={stat}, p-value={p}")
mean_diff = np.mean(stitchOT_results - jointOT_results)
print(f"Mean of paired differences (StitchOT - JointOT): {mean_diff} \n")

Wilcoxon Test for rel. thres. Accuracy between StitchOT and JointOT:
Wilcoxon test between StitchOT and JointOT: statistic=2722147.0, p-value=2.286009621298022e-108
Mean of paired differences (StitchOT - JointOT): -0.010290055115151141 

Wilcoxon Test for TV distance Rel. transition rates between StitchOT and JointOT:
Wilcoxon test between StitchOT and JointOT: statistic=3756495.0, p-value=8.492436798004769e-20
Mean of paired differences (StitchOT - JointOT): 0.0009699645271455443 

Wilcoxon Test for IEA between StitchOT and JointOT:
Wilcoxon test between StitchOT and JointOT: statistic=4241843.0, p-value=0.002673473957728669
Mean of paired differences (StitchOT - JointOT): 0.0005136046514139544 

Removed 3 rows with nans in either StitchOT or JointOT results for TV distance transition probs.
[np.float64(0.3197303498325273), np.float64(0.33943841289737964), np.float64(0.5447132531256003)]
Wilcoxon Test for TV distance transition probs. between StitchOT and JointOT:
Wilcoxon test betwee

In [10]:
# for stitchOT tau_a=0.5, do boxplots for all stats
plt.figure(figsize=(12, 6, "cm"), layout="constrained")
sns.violinplot(
    data=results[:, 2, 4:-2],
    palette=sns.color_palette("mako", 16)[6:],
    linewidth=1,
    inner="quart",
    cut=0,
)
# add mean as a white dot with grey edge
plt.scatter(
    x=np.arange(len(stats_names[4:-2])),
    y=[np.nanmean(results[:, 2, i]) for i in range(4, 14)],
    color="white",
    edgecolor="grey",
    s=4,
    zorder=10,
    linewidth=0.4,
)
plt.xticks(
    ticks=np.arange(len(stats_names[4:-2])), labels=stats_names[4:-2], rotation=90
)
# plt.ylabel('TV distance')
# plt.title("StitchOT $\\tau_a=0.5$ - Relative transition rates")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.gca().set_axisbelow(True)
plt.savefig(f"{figure_path}_violin_all_stats_stitchOT_tau_a_0.5.pdf")
plt.close()


# for jointOT tau_b=0.5, do boxplots for all stats
plt.figure(figsize=(12, 6, "cm"), layout="constrained")
sns.violinplot(
    data=results[:, 11, 4:-2],
    palette=sns.color_palette("mako", 16)[6:],
    linewidth=1,
    inner="quart",
    cut=0,
)
# add mean as a white dot with grey edge
plt.scatter(
    x=np.arange(len(stats_names[4:-2])),
    y=[np.nanmean(results[:, 11, i]) for i in range(4, 14)],
    color="white",
    edgecolor="grey",
    s=4,
    zorder=10,
    linewidth=0.4,
)
plt.xticks(
    ticks=np.arange(len(stats_names[4:-2])), labels=stats_names[4:-2], rotation=90
)
# plt.ylabel('TV distance')
# plt.title("StitchOT $\\tau_a=0.5$ - Relative transition rates")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.gca().set_axisbelow(True)
plt.savefig(f"{figure_path}_violin_all_stats_jointOT_tau_a_0.5.pdf")
plt.close()

In [11]:
accs = np.zeros((10000, 16, 2, 100))
for job_id in range(100):
    try:
        accs[job_id * 100 : (job_id + 1) * 100, :, :, :] = np.load(
            f"{results_path}/accs_{job_id}.npy"
        )
    except FileNotFoundError:
        continue

accs = accs[~np.all(accs == 0, axis=(1, 2, 3))]
print(f"Number of rows left: {accs.shape[0]}")

accs_stitchOT_inferred = accs[:, 2, 1, :]
accs_true = accs[:, 11, 0, :]
accs_JointOT_inferred = accs[:, 11, 1, :]
accs_random = accs[:, 8, 1, :]

avg_thres_accs_stitchOT_inferred = np.mean(accs_stitchOT_inferred[:, 0:50], axis=1)
avg_thres_accs_true = np.mean(accs_true[:, 0:50], axis=1)
avg_thres_accs_JointOT_inferred = np.mean(accs_JointOT_inferred[:, 0:50], axis=1)

rel_avg_thres_accs_stitchOT = avg_thres_accs_stitchOT_inferred / avg_thres_accs_true
rel_avg_thres_accs_JointOT = avg_thres_accs_JointOT_inferred / avg_thres_accs_true
# Plot both comparisons in one figure with two subplots
fig, axes = plt.subplots(1, 2, figsize=(8 * cm, 8 * cm))

# Top: thresholds 0-50 (already computed above)
sns.boxplot(
    data=[rel_avg_thres_accs_stitchOT, rel_avg_thres_accs_JointOT],
    showfliers=False,
    ax=axes[0],
)
axes[0].set_xticks(np.arange(2))
axes[0].set_xticklabels([method_names[2], method_names[11]], rotation=90)
axes[0].set_ylabel("Rel. thres. Accuracy (avg over thres. 0-50)")

# Bottom: thresholds 50-100
avg_thres_accs_stitchOT_inferred_50_100 = np.mean(
    accs_stitchOT_inferred[:, 50:100], axis=1
)
avg_thres_accs_true_50_100 = np.mean(accs_true[:, 50:100], axis=1)
avg_thres_accs_JointOT_inferred_50_100 = np.mean(
    accs_JointOT_inferred[:, 50:100], axis=1
)

rel_avg_thres_accs_stitchOT_50_100 = (
    avg_thres_accs_stitchOT_inferred_50_100 / avg_thres_accs_true_50_100
)
rel_avg_thres_accs_JointOT_50_100 = (
    avg_thres_accs_JointOT_inferred_50_100 / avg_thres_accs_true_50_100
)

sns.boxplot(
    data=[rel_avg_thres_accs_stitchOT_50_100, rel_avg_thres_accs_JointOT_50_100],
    showfliers=False,
    ax=axes[1],
)
axes[1].set_xticks(np.arange(2))
axes[1].set_xticklabels([method_names[2], method_names[11]], rotation=90)
axes[1].set_ylabel("Rel. thres. Accuracy (avg over thres. 50-100)")

plt.tight_layout()
plt.savefig(f"{figure_path}rel_avg_thres_accs_comparison.pdf")
plt.close()

Number of rows left: 4233


In [12]:
# Plot accuracy over thresholds for StitchOT tau_a=0.5 and JointOT tau_b=0.5 and ground truth
plt.figure(figsize=(5.5, 4.82, "cm"), layout="constrained")
plt.plot(
    np.arange(100),
    np.mean(accs_stitchOT_inferred, axis=0),
    label=method_names[2],
    c=colors_3[0],
)
plt.plot(
    np.arange(100),
    np.mean(accs_JointOT_inferred, axis=0),
    label=method_names[11],
    c=colors_3[1],
)
plt.plot(np.arange(100), np.mean(accs_true, axis=0), label="True", c="tab:green")
plt.plot(np.arange(100), np.mean(accs_random, axis=0), label="Random", c="tab:gray")
# also add fill between for the standard deviation
plt.fill_between(
    np.arange(100),
    np.mean(accs_stitchOT_inferred, axis=0) - np.std(accs_stitchOT_inferred, axis=0),
    np.mean(accs_stitchOT_inferred, axis=0) + np.std(accs_stitchOT_inferred, axis=0),
    alpha=0.2,
    color=colors_3[0],
    edgecolor=None,
)

plt.fill_between(
    np.arange(100),
    np.mean(accs_JointOT_inferred, axis=0) - np.std(accs_JointOT_inferred, axis=0),
    np.mean(accs_JointOT_inferred, axis=0) + np.std(accs_JointOT_inferred, axis=0),
    alpha=0.2,
    color=colors_3[1],
    edgecolor=None,
)

plt.fill_between(
    np.arange(100),
    np.mean(accs_true, axis=0) - np.std(accs_true, axis=0),
    np.mean(accs_true, axis=0) + np.std(accs_true, axis=0),
    alpha=0.2,
    color="tab:green",
    edgecolor=None,
)

plt.fill_between(
    np.arange(100),
    np.mean(accs_random, axis=0) - np.std(accs_random, axis=0),
    np.mean(accs_random, axis=0) + np.std(accs_random, axis=0),
    alpha=0.2,
    color=colors_3[2],
    edgecolor=None,
)

plt.xlabel(r"$\psi$ (Threshold [%])")
plt.ylabel(r"$\text{ACC}_{\psi}$")
plt.legend()
plt.savefig(f"{figure_path}thres_accs_comparison.pdf")
plt.close()

In [13]:
# Plot pointwise diffeerence, i.e. true - StitchOT and true - JointOT over thresholds
plt.figure(figsize=(5.5, 4.82, "cm"), layout="constrained")
plt.plot(
    np.arange(100),
    np.mean(accs_true - accs_stitchOT_inferred, axis=0),
    label=f"True - {method_names[2]}",
    c=colors_3[0],
)
plt.plot(
    np.arange(100),
    np.mean(accs_true - accs_JointOT_inferred, axis=0),
    label=f"True - {method_names[11]}",
    c=colors_3[1],
)
plt.plot(
    np.arange(100),
    np.mean(accs_true - accs_random, axis=0),
    label="True - Random",
    c=colors_3[2],
)
# also add fill between for the standard deviation
plt.fill_between(
    np.arange(100),
    np.mean(accs_true - accs_stitchOT_inferred, axis=0)
    - np.std(accs_true - accs_stitchOT_inferred, axis=0),
    np.mean(accs_true - accs_stitchOT_inferred, axis=0)
    + np.std(accs_true - accs_stitchOT_inferred, axis=0),
    alpha=0.2,
    color=colors_3[0],
    edgecolor=None,
)
plt.fill_between(
    np.arange(100),
    np.mean(accs_true - accs_JointOT_inferred, axis=0)
    - np.std(accs_true - accs_JointOT_inferred, axis=0),
    np.mean(accs_true - accs_JointOT_inferred, axis=0)
    + np.std(accs_true - accs_JointOT_inferred, axis=0),
    alpha=0.2,
    color=colors_3[1],
    edgecolor=None,
)
plt.fill_between(
    np.arange(100),
    np.mean(accs_true - accs_random, axis=0) - np.std(accs_true - accs_random, axis=0),
    np.mean(accs_true - accs_random, axis=0) + np.std(accs_true - accs_random, axis=0),
    alpha=0.2,
    color=colors_3[2],
    edgecolor=None,
)
# dotted horziontal line with alpha 0.7
plt.axhline(0, color=colors_3[2], linestyle="--", alpha=0.5, linewidth=0.7)
plt.xlabel(r"$\psi$ (Threshold [%])")
plt.ylabel(r"$\text{ACC}_{\psi}$ Difference")
plt.legend()
plt.savefig(f"{figure_path}thres_accs_difference.pdf")
plt.close()